# Human evaluation
We will do standard tea leaves evaluation on the following representative models
+ 4-gram: base model
+ 4-gram: tensor train variant
+ 4-gram: tensor train variant with 300 rank

In [7]:
from tensormet.tucker_tensor import TuckerDecomposition
import random
from tensormet.utils import select_gpu
dev = select_gpu()
random.seed(1)
tk_dict = {}
tk =  TuckerDecomposition.load_from_disk()
# tk.update_from_path()
tk_dict["4-gram_base"] = tk

tk_tt = TuckerDecomposition.load_from_disk(decomposition="tt")
tk_tt.update_from_path()
tk_dict["4-gram_tt"] = tk_tt
# tk_highdim = TuckerDecomposition.load_from_disk(decomposition="tt", dims=200000)
# tk_highdim.update_from_path()
# tk_dict["4-gram_highdim"] = tk_highdim
# tk_dict["4-gram_300r"] = TuckerDecomposition.load_from_disk(decomposition="tt", rank=300)

Pinned to GPU(s): [3]


In [8]:
from tensormet.experimental.tealeaves_human import run_widget, open_session
# Build every trial for every model in one go (one argsort per factor), so no
# dimension costs anything once annotation starts. Files that already exist are
# resumed rather than rebuilt, and their trials are checked against the model.
sessions = {annotator: {
    name: open_session(tk, annotator=annotator, num_dim_words=4, seed=1, name=name)
    for name, tk in tk_dict.items()
} for annotator in ["tim", "stef", "maria"]
}
for name, s in sessions["stef"].items():
    print(f"{name:8s} {s.n_done:>3}/{s.n_tasks} answered   {s.path.name}")

Resuming 4-gram_base_w1_k4_seed1_tim.jsonl: 100/100 answered.
Resuming 4-gram_tt_w1_k4_seed1_tim.jsonl: 100/100 answered.
Resuming 4-gram_base_w1_k4_seed1_stef.jsonl: 100/100 answered.
Resuming 4-gram_tt_w1_k4_seed1_stef.jsonl: 100/100 answered.
Resuming 4-gram_base_w1_k4_seed1_maria.jsonl: 100/100 answered.
Resuming 4-gram_tt_w1_k4_seed1_maria.jsonl: 100/100 answered.
4-gram_base 100/100 answered   4-gram_base_w1_k4_seed1_stef.jsonl
4-gram_tt 100/100 answered   4-gram_tt_w1_k4_seed1_stef.jsonl


In [9]:
# One click per dimension, no model call in between. Every answer is on disk
# before the next trial is drawn, so interrupting the kernel costs nothing --
# re-run this cell (or the one above) to carry on where you stopped.
# run_widget(sessions["lab"]["4-gram_highdim"])

In [10]:
# run_widget(sessions["stef"]["4-gram_tt"])

In [11]:
# run_widget(sessions["stef"]["4-gram_highdim"])

# Preparing human evaluation for colleagues

In [12]:
for name, s in sessions["stef"].items():
    print(f"{name:8s} {s.summary()}")

4-gram_base {'human_dim_consistency': 0.6992, 'human_dim_consistency_raw': 0.92, 'human_dim_consistency_diversity': 0.76, 'n_correct': 92, 'n_scored': 100, 'n_answered': 100, 'n_skipped': 0, 'n_tasks': 100, 'median_seconds': 3.3}
4-gram_tt {'human_dim_consistency': 0.7343999999999999, 'human_dim_consistency_raw': 0.96, 'human_dim_consistency_diversity': 0.765, 'n_correct': 96, 'n_scored': 100, 'n_answered': 99, 'n_skipped': 1, 'n_tasks': 100, 'median_seconds': 2.94}


In [13]:
for name, s in sessions["tim"].items():
    print(f"{name:8s} {s.summary()}")

4-gram_base {'human_dim_consistency': 0.5928, 'human_dim_consistency_raw': 0.78, 'human_dim_consistency_diversity': 0.76, 'n_correct': 78, 'n_scored': 100, 'n_answered': 93, 'n_skipped': 7, 'n_tasks': 100, 'median_seconds': 5.18}
4-gram_tt {'human_dim_consistency': 0.63495, 'human_dim_consistency_raw': 0.83, 'human_dim_consistency_diversity': 0.765, 'n_correct': 83, 'n_scored': 100, 'n_answered': 99, 'n_skipped': 1, 'n_tasks': 100, 'median_seconds': 4.53}


In [14]:
for name, s in sessions["maria"].items():
    print(f"{name:8s} {s.summary()}")

4-gram_base {'human_dim_consistency': 0.6764, 'human_dim_consistency_raw': 0.89, 'human_dim_consistency_diversity': 0.76, 'n_correct': 89, 'n_scored': 100, 'n_answered': 100, 'n_skipped': 0, 'n_tasks': 100, 'median_seconds': 8.53}
4-gram_tt {'human_dim_consistency': 0.6732, 'human_dim_consistency_raw': 0.88, 'human_dim_consistency_diversity': 0.765, 'n_correct': 88, 'n_scored': 100, 'n_answered': 100, 'n_skipped': 0, 'n_tasks': 100, 'median_seconds': 10.59}


# Interpretable baselines: NNSE and SINr
The same task on two released interpretable embeddings, for the same annotators:
+ `nnse_300`, `nnse_1000`: NNSE (Murphy et al., 2012), the released depDoc models
+ `sinr_bnc`: SINr (Prouteau et al., 2021), the BNC model shipped in the SINr repo (lemmas of nouns, verbs and adjectives only)

Set-up, so the numbers stay comparable with the sets above:
+ each dimension is ranked over the words of `4-gram_base` (all-zero rows dropped, as in the POLAR suite)
+ 100 dimensions per baseline, sampled with seed 1 from those with at least 4 non-zero words, so every trial shows 4 top words. `human_eval_results/baseline_dims.json` lists which ones: session dimension `i` is original dimension `dims[i]`
+ k = 4 and trial seed 1, as above. The intruder pool is computed over the 100 sampled dimensions, since the wrapped matrix holds only those

Files and readers come from the POLAR suite (`../polar/method_baselines.py`); provenance in `../README.md`.

In [15]:
import json
import random
import sys
from pathlib import Path

import numpy as np

import method_baselines as mb
from glove_baseline import _as_baseline
from tensormet.experimental.tealeaves_human import DEFAULT_RESULTS_DIR, open_session

like = tk_dict["4-gram_tt"]
ours = set(like.vocab[f"vocab_{like.roles[0]}"])
released = {"nnse_300": mb.read_txt(mb.NNSE_DIR / mb.NNSE_FILE.format(300)),
            "nnse_1000": mb.read_txt(mb.NNSE_DIR / mb.NNSE_FILE.format(1000)),
            "sinr_bnc": mb.read_sinr(mb.SINR_PATH),
            "glove_100": mb.read_glove(mb.GLOVE_PATH, keep=ours)}  # GloVe 2024 100d: all 100 dimensions
baselines, baseline_dims = {}, {}
for name, (words, E) in released.items():
    rows = [i for i, w in enumerate(words) if w in ours and E[i].any()]  # our words; all-zero rows = unknown
    E = E[rows]
    usable = np.flatnonzero((E > 0).sum(0) >= 4).tolist()  # a trial needs 4 top words
    dims = sorted(random.Random(1).sample(usable, 100))
    baseline_dims[name] = dims  # session dimension i = original dimension dims[i]
    baselines[name] = _as_baseline([words[i] for i in rows], E[:, dims], like.roles, name)
    print(f"{name}: {len(rows)} of our {len(ours)} words, {len(usable)} of {E.shape[1]} dimensions usable, 100 sampled")
(DEFAULT_RESULTS_DIR / "baseline_dims.json").write_text(json.dumps(baseline_dims))

for annotator in ["stef", "tim", "maria"]:
    for name, emb in baselines.items():
        sessions.setdefault(annotator, {})[name] = open_session(emb, annotator=annotator, num_dim_words=4,
                                                                seed=1, name=name)

35362 vectors of dim 300 read from depDocNNSE300.tab.zip
35529 vectors of dim 1000 read from depDocNNSE1000.tab.zip
28751 vectors of dim 1431 read from sinrvec_bnc.pk
9987 vectors of dim 100 read from wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt (13 of the model's words missing)
nnse_300: 9286 of our 10000 words, 300 of 300 dimensions usable, 100 sampled
nnse_1000: 9444 of our 10000 words, 1000 of 1000 dimensions usable, 100 sampled
sinr_bnc: 6004 of our 10000 words, 1431 of 1431 dimensions usable, 100 sampled
glove_100: 9987 of our 10000 words, 100 of 100 dimensions usable, 100 sampled
Resuming nnse_300_w1_k4_seed1_stef.jsonl: 100/100 answered.
Resuming nnse_1000_w1_k4_seed1_stef.jsonl: 0/100 answered.
Resuming sinr_bnc_w1_k4_seed1_stef.jsonl: 0/100 answered.
Resuming glove_100_w1_k4_seed1_stef.jsonl: 100/100 answered.
Resuming nnse_300_w1_k4_seed1_tim.jsonl: 100/100 answered.
Resuming nnse_1000_w1_k4_seed1_tim.jsonl: 0/100 answered.
Resuming sinr_bn

In [16]:
run_widget(sessions["stef"]["nnse_300"])

In [17]:
run_widget(sessions["stef"]["glove_100"])

In [18]:
run_widget(sessions["stef"]["sinr_bnc"])

In [19]:
import pandas as pd

# Every session file on disk (all sets, all annotators), read afresh: raw accuracy, then trials answered.
summary = pd.DataFrame([{"set": s.meta["model"], "annotator": s.meta["annotator"], **s.summary()}
                        for s in (open_session(path=p, verbose=False)
                                  for p in sorted(DEFAULT_RESULTS_DIR.glob("*.jsonl")))])
display(summary.pivot(index="set", columns="annotator", values="human_dim_consistency_raw").round(3))
display(summary.pivot(index="set", columns="annotator", values="n_scored"))

annotator,lab,maria,stef,tim
set,,,,
4-gram_base,NaN,0.89,0.92,0.78
4-gram_highdim,1.0,0.00,0.99,0.00
4-gram_tt,NaN,0.88,0.96,0.83
glove_100,NaN,0.43,0.39,0.37
nnse_1000,NaN,0.00,0.00,0.00
nnse_300,NaN,0.91,0.84,0.84
sinr_bnc,NaN,0.00,0.00,0.00


annotator,lab,maria,stef,tim
set,,,,
4-gram_base,NaN,100.0,100.0,100.0
4-gram_highdim,3.0,0.0,100.0,0.0
4-gram_tt,NaN,100.0,100.0,100.0
glove_100,NaN,100.0,100.0,100.0
nnse_1000,NaN,0.0,0.0,0.0
nnse_300,NaN,100.0,100.0,100.0
sinr_bnc,NaN,0.0,0.0,0.0


In [20]:
from tensormet.experimental.tealeaves_human import compare_with_judge
# Human vs. LLM on the very same dimensions. Same role, seed and k -> identical
# trials; compare_with_judge verifies that per dimension instead of trusting it.
from tensormet.judge import DimConsistencyJudge

session = sessions["stef"]["4-gram_base"]
judge = DimConsistencyJudge(num_dim_words=session.meta["num_dim_words"])
judge_out = judge.score(tk_dict["4-gram_base"], intruder_choice_option="teaLeaves",
                        seed=session.meta["seed"], role=session.meta["role"],
                        return_details=True)
judge.unload()

comparison = compare_with_judge(session, judge_out)
print({k: v for k, v in comparison.items() if k != "rows"})
for row in comparison["rows"][:10]:
    print(f"dim {row['dim']:>3}  outlier {row['outlier']!r:>14}  "
          f"human {row['human']!r:>14}  judge {row['judge']!r:>14}")

Loading dimension-consistency judge 'Qwen/Qwen3.5-2B' (fp16)...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 3.51 GB GPU memory.
{'n_compared': 100, 'n_skipped_excluded': 0, 'human_accuracy': 0.92, 'judge_accuracy': 0.58, 'agreement': 0.58}
dim   0  outlier 'conversation'  human 'conversation'  judge 'conversation'
dim   1  outlier    'signifies'  human        'india'  judge        'least'
dim   2  outlier  'facilitates'  human  'facilitates'  judge  'facilitates'
dim   3  outlier   'properties'  human          'how'  judge   'properties'
dim   4  outlier       'hawaii'  human       'hawaii'  judge       'hawaii'
dim   5  outlier       'advice'  human       'advice'  judge    'regarding'
dim   6  outlier  'procurement'  human  'procurement'  judge           'at'
dim   7  outlier       'ticket'  human       'ticket'  judge        'while'
dim   8  outlier     'requires'  human     'requires'  judge       'damage'
dim   9  outlier        'drove'  human        'drove'  judge        'drove'


In [21]:
from tensormet.experimental.tealeaves_human import compare_with_judge
# Human vs. LLM on the very same dimensions. Same role, seed and k -> identical
# trials; compare_with_judge verifies that per dimension instead of trusting it.
from tensormet.judge import DimConsistencyJudge

session = sessions["tim"]["4-gram_base"]
judge = DimConsistencyJudge(num_dim_words=session.meta["num_dim_words"])
judge_out = judge.score(tk_dict["4-gram_base"], intruder_choice_option="teaLeaves",
                        seed=session.meta["seed"], role=session.meta["role"],
                        return_details=True)
judge.unload()

comparison = compare_with_judge(session, judge_out)
print({k: v for k, v in comparison.items() if k != "rows"})
for row in comparison["rows"][:10]:
    print(f"dim {row['dim']:>3}  outlier {row['outlier']!r:>14}  "
          f"human {row['human']!r:>14}  judge {row['judge']!r:>14}")

Loading dimension-consistency judge 'Qwen/Qwen3.5-2B' (fp16)...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 3.51 GB GPU memory.
{'n_compared': 93, 'n_skipped_excluded': 7, 'human_accuracy': 0.8387096774193549, 'judge_accuracy': 0.6021505376344086, 'agreement': 0.6344086021505376}
dim   0  outlier 'conversation'  human 'conversation'  judge 'conversation'
dim   3  outlier   'properties'  human   'properties'  judge   'properties'
dim   4  outlier       'hawaii'  human       'hawaii'  judge       'hawaii'
dim   5  outlier       'advice'  human       'advice'  judge    'regarding'
dim   6  outlier  'procurement'  human  'procurement'  judge           'at'
dim   7  outlier       'ticket'  human       'ticket'  judge        'while'
dim   8  outlier     'requires'  human     'requires'  judge       'damage'
dim   9  outlier        'drove'  human        'drove'  judge        'drove'
dim  10  outlier         'shop'  human         'shop'  judge       'access'
dim  11  outlier      'visitor'  human      'visitor'  judge      'visitor'


# Is the judge a usable heuristic for human accuracy?

Only `4-gram_base` and `4-gram_tt` have completed human sessions (2 models x 2
annotators = 4 data points) -- far too few to correlate *model-level* accuracy
across models. Instead we pool the per-dimension correctness that
`compare_with_judge` already gives us (human correct vs. judge correct on the
same trial) across all four sessions, and check whether the judge and the
humans tend to succeed/fail on the same dimensions. A positive, significant
correlation there is evidence the judge tracks human judgment at the item
level, which is what aggregate judge accuracy would need to be a valid proxy
for model-level human accuracy.

In [22]:
from tensormet.experimental.tealeaves_human import compare_with_judge, judge_session
from tensormet.judge import DimConsistencyJudge
from scipy.stats import pearsonr

# The judge scores the trials stored in each session file
# Scoring is deterministic and all annotators of a model share the same trials, so
# dim_consistency_raw is one fixed number per model. (compare_with_judge's own
# accuracy DOES vary by annotator, because it's restricted to the dimensions that
# annotator answered, skips excluded -- a subset artifact, so we read the raw
# score off judge_out instead.)
judge = DimConsistencyJudge(num_dim_words=4, diversity_aware=False,)
comparisons, judge_raw = {}, {}
for model_name in ["4-gram_base", "4-gram_tt", "nnse_300", "glove_100"]:
    for annotator in ["tim", "stef", "maria"]:
        session = sessions[annotator][model_name]
        if session.n_done == 0:
            continue
        judge_out = judge_session(judge, session)
        judge_raw[model_name] = judge_out["dim_consistency_raw"]
        comparisons[(model_name, annotator)] = compare_with_judge(session, judge_out)
judge.unload()

for model_name in ["4-gram_base", "4-gram_tt", "nnse_300", "glove_100"]:
    for annotator in ["tim", "stef", "maria"]:
        human_raw = sessions[annotator][model_name].summary()["human_dim_consistency_raw"]
        print(f"{model_name:12s} {annotator:5s} human_raw={human_raw:.3f}  judge_raw={judge_raw[model_name]:.3f}")

# Item-level correlation: on the same dimension, does the judge being right
# track the human being right? (Pearson on 0/1 data = phi coefficient. Unaffected
# by raw vs. diversity-aware, since that multiplier only rescales the aggregate.)
human_correct = [row["human_correct"] for c in comparisons.values() for row in c["rows"]]
judge_correct = [row["judge_correct"] for c in comparisons.values() for row in c["rows"]]
r, p = pearsonr(human_correct, judge_correct)
print(f"\nitem-level phi correlation: r={r:.3f}, p={p:.3g}, n={len(human_correct)}")

Loading dimension-consistency judge 'Qwen/Qwen3.5-2B' (fp16)...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 3.51 GB GPU memory.
4-gram_base  tim   human_raw=0.780  judge_raw=0.580
4-gram_base  stef  human_raw=0.920  judge_raw=0.580
4-gram_base  maria human_raw=0.890  judge_raw=0.580
4-gram_tt    tim   human_raw=0.830  judge_raw=0.510
4-gram_tt    stef  human_raw=0.960  judge_raw=0.510
4-gram_tt    maria human_raw=0.880  judge_raw=0.510
nnse_300     tim   human_raw=0.840  judge_raw=0.580
nnse_300     stef  human_raw=0.840  judge_raw=0.580
nnse_300     maria human_raw=0.910  judge_raw=0.580
glove_100    tim   human_raw=0.370  judge_raw=0.300
glove_100    stef  human_raw=0.390  judge_raw=0.300
glove_100    maria human_raw=0.430  judge_raw=0.300

item-level phi correlation: r=0.228, p=3.72e-15, n=1161


## Judge ablation: base model × prompt
One row per variant, same pooled item-level phi as above.

In [23]:
import itertools
import pandas as pd
from tensormet.experimental.tealeaves_human import compare_with_judge, judge_session
from tensormet.judge import DEFAULT_JUDGE_MODEL, DEFAULT_JUDGE_MESSAGES

# Same pooled item-level check as above, for every (base model, prompt) pair.
# "{listing}" in a message is filled with the candidates. Add an entry and re-run:
# finished pairs are cached by (model, messages) and not recomputed.
FEW_SHOT = [
    {"role": "system", "content": "You are a helpful assistant tasked with identifying the outlier in a list of words."},
    {"role": "user", "content": "Which word does not belong? apple, banana, orange, car, grape."},
    {"role": "assistant", "content": "car"},
    {"role": "user", "content": "Which word does not belong? like, love, message, adore, hate."},
    {"role": "assistant", "content": "message"},
    {"role": "user", "content": "Which word does not belong? {listing}. Answer with only the word."},
]
STAMMBACH = [
        {"role": "system",
         "content": "You are a helpful assistant evaluating the top words of a topic model output "
        "for a given topic. Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user",
         "content": "{listing}"},
    ]

STAMMBACH_ADAPTED = [{"role": "system",
         "content":
             "You are a helpful assistant evaluating a model with interpretable dimensions. "
             "Dimensions are interpretable because its top words share a meaning, "
             "grammatical form, or the same role in a sentence. "
             "Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user",
         "content": "{listing}"},]

STAMMBACH_ADAPTED_FEWSHOT = [{"role": "system",
         "content":
             "You are a helpful assistant evaluating a model with interpretable dimensions. "
             "Dimensions are interpretable because its top words share a meaning, "
             "grammatical form, or the same role in a sentence. "
             "Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user", "content": "Which word does not belong? walked, jumped, ran, sang, table."},
        {"role": "assistant", "content": "table"},
        {"role": "user", "content": "Which word does not belong? of, in, on, at, running."},
        {"role": "assistant", "content": "running"},
        {"role": "user", "content": "Which word does not belong? teach, teaches, teaching, taught, garden."},
        {"role": "assistant", "content": "garden"},
        {"role": "user", "content": "Which word does not belong? {listing}. Answer with only the word."},]

MODELS = {
    "qwen3.5-2b": DEFAULT_JUDGE_MODEL,
    "qwen3.5-4b": "Qwen/Qwen3.5-4B",
}
PROMPTS = {
    "default": DEFAULT_JUDGE_MESSAGES,
    "few-shot": FEW_SHOT,
    "stammbach": STAMMBACH,
    # "stammbach-adapted": STAMMBACH_ADAPTED,
    # "stammbach-adapted-few-shot": STAMMBACH_ADAPTED_FEWSHOT,
}
# Model is the outer loop, so each base model is loaded once for all its prompts.
variants = {f"{m} | {p}": (MODELS[m], PROMPTS[p])
            for m, p in itertools.product(MODELS, PROMPTS)}

judge_cache = globals().get("judge_cache", {})
judge = None
for label, (model_name, messages) in variants.items():
    key = (model_name, repr(messages))
    if key in judge_cache:
        continue
    if judge is None or judge.model_name != model_name:
        if judge is not None:
            judge.unload()
        judge = DimConsistencyJudge(model_name, num_dim_words=4, diversity_aware=False)
    judge.messages = messages
    rows, res = [], {}
    for model in ["4-gram_base", "4-gram_tt", "nnse_300", "glove_100"]:
        for annotator in ["tim", "stef", "maria"]:
            session = sessions[annotator][model]
            out = judge_session(judge, session)  # the trials this annotator saw
            res[f"judge_raw {model}"] = out["dim_consistency_raw"]
            rows += compare_with_judge(session, out)["rows"]
    r, p = pearsonr([x["human_correct"] for x in rows], [x["judge_correct"] for x in rows])
    judge_cache[key] = {**res, "agreement": sum(x["agree"] for x in rows) / len(rows),
                        "phi": r, "p": p, "n": len(rows)}
if judge is not None:
    judge.unload()

pd.DataFrame({label: judge_cache[(m, repr(msgs))]
              for label, (m, msgs) in variants.items()}).T.style.format("{:.3g}")

Loading dimension-consistency judge 'Qwen/Qwen3.5-2B' (fp16)...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 3.51 GB GPU memory.
Loading dimension-consistency judge 'Qwen/Qwen3.5-4B' (fp16)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 7.93 GB GPU memory.


,judge_raw 4-gram_base,judge_raw 4-gram_tt,judge_raw nnse_300,judge_raw glove_100,agreement,phi,p,n
qwen3.5-2b | default,0.58,0.51,0.58,0.3,0.485,0.228,3.72e-15,1.16e+03
qwen3.5-2b | few-shot,0.59,0.5,0.52,0.28,0.469,0.241,7.89e-17,1.16e+03
qwen3.5-2b | stammbach,0.71,0.63,0.26,0.17,0.425,0.166,1.31e-08,1.16e+03
qwen3.5-4b | default,0.82,0.84,0.77,0.43,0.67,0.353,2.01e-35,1.16e+03
qwen3.5-4b | few-shot,0.74,0.73,0.77,0.44,0.638,0.343,1.9e-33,1.16e+03
qwen3.5-4b | stammbach,0.74,0.64,0.28,0.23,0.452,0.195,2.27e-11,1.16e+03


In [ ]:
import itertools
import pandas as pd
from tensormet.experimental.tealeaves_human import compare_with_judge, judge_session
from tensormet.judge import DEFAULT_JUDGE_MODEL, DEFAULT_JUDGE_MESSAGES

# Same pooled item-level check as above, for every (base model, prompt) pair.
# "{listing}" in a message is filled with the candidates. Add an entry and re-run:
# finished pairs are cached by (model, messages) and not recomputed.
FEW_SHOT = [
    {"role": "system", "content": "You are a helpful assistant tasked with identifying the outlier in a list of words."},
    {"role": "user", "content": "Which word does not belong? apple, banana, orange, car, grape."},
    {"role": "assistant", "content": "car"},
    {"role": "user", "content": "Which word does not belong? like, love, message, adore, hate."},
    {"role": "assistant", "content": "message"},
    {"role": "user", "content": "Which word does not belong? {listing}. Answer with only the word."},
]
STAMMBACH = [
        {"role": "system",
         "content": "You are a helpful assistant evaluating the top words of a topic model output "
        "for a given topic. Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user",
         "content": "{listing}"},
    ]

STAMMBACH_ADAPTED = [{"role": "system",
         "content":
             "You are a helpful assistant evaluating a model with interpretable dimensions. "
             "Dimensions are interpretable because its top words share a meaning, "
             "grammatical form, or the same role in a sentence. "
             "Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user",
         "content": "{listing}"},]

STAMMBACH_ADAPTED_FEWSHOT = [{"role": "system",
         "content":
             "You are a helpful assistant evaluating a model with interpretable dimensions. "
             "Dimensions are interpretable because its top words share a meaning, "
             "grammatical form, or the same role in a sentence. "
             "Select which word is the least related to all other words. "
         "If multiple words do not fit, choose the word that is most out of place."},
        {"role": "user", "content": "Which word does not belong? walked, jumped, ran, sang, table."},
        {"role": "assistant", "content": "table"},
        {"role": "user", "content": "Which word does not belong? of, in, on, at, running."},
        {"role": "assistant", "content": "running"},
        {"role": "user", "content": "Which word does not belong? teach, teaches, teaching, taught, garden."},
        {"role": "assistant", "content": "garden"},
        {"role": "user", "content": "Which word does not belong? {listing}. Answer with only the word."},]

MODELS = {
    "qwen3.5-9b": "Qwen/Qwen3.5-9B",
}
PROMPTS = {
    "default": DEFAULT_JUDGE_MESSAGES,
    "few-shot": FEW_SHOT,
    "stammbach": STAMMBACH,
    # "stammbach-adapted": STAMMBACH_ADAPTED,
    # "stammbach-adapted-few-shot": STAMMBACH_ADAPTED_FEWSHOT,
}
# Model is the outer loop, so each base model is loaded once for all its prompts.
variants = {f"{m} | {p}": (MODELS[m], PROMPTS[p])
            for m, p in itertools.product(MODELS, PROMPTS)}

judge_cache = globals().get("judge_cache", {})
judge = None
for label, (model_name, messages) in variants.items():
    key = (model_name, repr(messages))
    if key in judge_cache:
        continue
    if judge is None or judge.model_name != model_name:
        if judge is not None:
            judge.unload()
        judge = DimConsistencyJudge(model_name, num_dim_words=4, diversity_aware=False)
    judge.messages = messages
    rows, res = [], {}
    for model in ["4-gram_base", "4-gram_tt", "nnse_300", "glove_100"]:
        for annotator in ["tim", "stef", "maria"]:
            session = sessions[annotator][model]
            out = judge_session(judge, session)  # the trials this annotator saw
            res[f"judge_raw {model}"] = out["dim_consistency_raw"]
            rows += compare_with_judge(session, out)["rows"]
    r, p = pearsonr([x["human_correct"] for x in rows], [x["judge_correct"] for x in rows])
    judge_cache[key] = {**res, "agreement": sum(x["agree"] for x in rows) / len(rows),
                        "phi": r, "p": p, "n": len(rows)}
if judge is not None:
    judge.unload()

pd.DataFrame({label: judge_cache[(m, repr(msgs))]
              for label, (m, msgs) in variants.items()}).T.style.format("{:.3g}")

Loading dimension-consistency judge 'Qwen/Qwen3.5-9B' (fp16)...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 16.68 GB GPU memory.


## Judge as one more annotator
Each rater's mean agreement with every *other* human (leave-one-out). The humans'
mean is the ceiling; `*_ratio` = judge / human (1.0 = as good as an extra
annotator). Logic lives in `judge_ablation.py`, shared with the
H100 runs.
```
  ┌──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │        Column        │                                                                                   What it is                                                                                   │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ judge_phi            │ The judge's mean φ with each human. φ is the correlation of right/wrong per trial, as explained above.                                                                         │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ human_phi            │ For each human, their mean φ with the other humans, then averaged over the humans. This is the ceiling: how well people agree with each other.                                 │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ phi_ratio            │ judge_phi / human_phi. 1.0 means the judge's failures line up with the humans' failures as well as the humans' line up with each other's. Below 1 means it's a worse stand-in. │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ judge_pick_agreement │ The judge's mean fraction of trials where it clicked the same word as a human.                                                                                                 │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ human_pick_agreement │ The same for human–human pairs, averaged: the ceiling for picks.                                                                                                               │
  ├──────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │ pick_agreement_ratio │ judge_pick_agreement / human_pick_agreement.                                                                                                                                   │
  └──────────────────────┴────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

In [21]:
import gc

import torch

# Free what earlier judge runs left behind before loading the next judge models.
gc.collect()
torch.cuda.empty_cache()
if judge is not None:
    judge.unload()

In [24]:
import itertools
import pandas as pd
from tensormet.experimental.tealeaves_human import DEFAULT_RESULTS_DIR
from tensormet.judge import DEFAULT_JUDGE_MODEL, DimConsistencyJudge
from judge_ablation import (PROMPTS, judge_trial_set, load_trial_sets,
                                              print_human_agreement, row_for)

# Sessions are re-read from disk, so a new annotator shows up on re-run without
# judging anything again; only an unseen (model, prompt, trial set) loads a model.
# Extra prompt to try: PROMPTS = {**PROMPTS, "mine": [...]}, "{listing}" = candidates.
MODELS = [DEFAULT_JUDGE_MODEL, "Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-9B"]

trial_sets = load_trial_sets(DEFAULT_RESULTS_DIR, "*.jsonl")
print_human_agreement(trial_sets)

judge_recs = globals().get("judge_recs", {})
judge = None
for model_name, (prompt_name, messages) in itertools.product(MODELS, PROMPTS.items()):
    rec = judge_recs.get((model_name, prompt_name))
    if rec is None or rec["messages"] != messages:
        rec = judge_recs[(model_name, prompt_name)] = {
            "model": model_name, "prompt": prompt_name, "messages": messages, "trial_sets": {}}
    missing = [fp for fp in trial_sets if fp not in rec["trial_sets"]]
    if not missing:
        continue
    if judge is None or judge.model_name != model_name:
        if judge is not None:
            judge.unload()
        judge = DimConsistencyJudge(model_name)
    judge.messages = messages
    for fp in missing:
        rec["trial_sets"][fp] = judge_trial_set(judge, trial_sets[fp]["tasks"])
if judge is not None:
    judge.unload()

rows = [row_for(judge_recs[key], trial_sets) for key in itertools.product(MODELS, PROMPTS)]
(pd.DataFrame(rows).set_index(["model", "prompt"])
   .sort_values("phi_ratio", ascending=False).style.format("{:.3g}"))

  4-gram_base_w1_k4_seed1_maria.jsonl: 100/100 answered
  4-gram_base_w1_k4_seed1_stef.jsonl: 100/100 answered
  4-gram_base_w1_k4_seed1_tim.jsonl: 100/100 answered
  4-gram_tt_w1_k4_seed1_maria.jsonl: 100/100 answered
  4-gram_tt_w1_k4_seed1_stef.jsonl: 100/100 answered
  4-gram_tt_w1_k4_seed1_tim.jsonl: 100/100 answered
  maria vs stef: phi=0.406  pick_agreement=0.905  n=199
  maria vs tim: phi=0.518  pick_agreement=0.870  n=192
  stef vs tim: phi=0.341  pick_agreement=0.853  n=191
  maria vs other humans: phi=0.462  pick_agreement=0.887
  stef vs other humans: phi=0.373  pick_agreement=0.879
  tim vs other humans: phi=0.429  pick_agreement=0.862
Loading dimension-consistency judge 'Qwen/Qwen3.5-2B' (fp16)...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 3.51 GB GPU memory.
Loading dimension-consistency judge 'Qwen/Qwen3.5-4B' (fp16)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 7.93 GB GPU memory.
Loading dimension-consistency judge 'Qwen/Qwen3.5-9B' (fp16)...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Judge model loaded on cuda:0 using 16.68 GB GPU memory.


# Including non-Qwen models
| model                               | prompt    | judge_phi | human_phi | phi_ratio | judge_pick_agreement | human_pick_agreement | pick_agreement_ratio | judge_acc 4-gram_base | judge_acc 4-gram_tt |
| ----------------------------------- | --------- | --------- | --------- | --------- | -------------------- | -------------------- | -------------------- | --------------------- | ------------------- |
| Qwen/Qwen3.5-9B                     | default   | 0.332     | 0.422     | 0.787     | 0.831                | 0.876                | 0.948                | 0.82                  | 0.88                |
| Qwen/Qwen3.5-9B                     | few-shot  | 0.33      | 0.422     | 0.783     | 0.79                 | 0.876                | 0.902                | 0.78                  | 0.81                |
| Qwen/Qwen3.5-4B                     | default   | 0.309     | 0.422     | 0.733     | 0.799                | 0.876                | 0.912                | 0.82                  | 0.84                |
| Qwen/Qwen3.5-4B                     | few-shot  | 0.258     | 0.422     | 0.613     | 0.716                | 0.876                | 0.817                | 0.74                  | 0.73                |
| allenai/OLMo-2-1124-7B-Instruct     | few-shot  | 0.215     | 0.422     | 0.509     | 0.548                | 0.876                | 0.626                | 0.5                   | 0.53                |
| Qwen/Qwen3.5-2B                     | default   | 0.214     | 0.422     | 0.508     | 0.547                | 0.876                | 0.624                | 0.58                  | 0.51                |
| microsoft/Phi-4-mini-instruct       | default   | 0.199     | 0.422     | 0.473     | 0.599                | 0.876                | 0.684                | 0.56                  | 0.62                |
| microsoft/Phi-4-mini-instruct       | few-shot  | 0.149     | 0.422     | 0.353     | 0.413                | 0.876                | 0.471                | 0.4                   | 0.41                |
| ibm-granite/granite-3.3-8b-instruct | default   | 0.143     | 0.422     | 0.34      | 0.474                | 0.876                | 0.541                | 0.49                  | 0.43                |
| Qwen/Qwen3.5-2B                     | few-shot  | 0.136     | 0.422     | 0.324     | 0.519                | 0.876                | 0.593                | 0.59                  | 0.5                 |
| ibm-granite/granite-3.3-8b-instruct | few-shot  | 0.135     | 0.422     | 0.321     | 0.48                 | 0.876                | 0.549                | 0.47                  | 0.47                |
| allenai/OLMo-2-1124-7B-Instruct     | default   | 0.111     | 0.422     | 0.263     | 0.467                | 0.876                | 0.533                | 0.51                  | 0.43                |